# HAVEN — RAG Pipeline Exploration

End-to-end walkthrough of the Retrieval-Augmented Generation stack:

1. **PDF extraction** — text extraction + OCR fallback
2. **Chunking** — split into ~200-token overlapping chunks
3. **Embedding** — `all-MiniLM-L6-v2` via sentence-transformers (fully local)
4. **FAISS index** — build and persist to disk
5. **Retriever** — semantic search over the knowledge base
6. **LLM backends** — Ollama (local) and Groq (free deployment)
7. **Full pipeline** — query → retrieved chunks → cited answer
8. **Gap-aware queries** — answers personalised to your kit inventory
9. **Multi-source comparison** — which guides contribute to which queries

Source documents: 4 official EU government emergency preparedness guides (CZ, SE, BE, NL)

## 0. Setup

%pip install pymupdf pytesseract pillow faiss-cpu --verbose
# Tesseract system package (run once in terminal, not here):
#   macOS:  brew install tesseract
#   Ubuntu: sudo apt install tesseract-ocr

%pip install sentence-transformers --verbose

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path(".")
PDF_DIR      = PROJECT_ROOT / "data" / "pdfs"
FAISS_DIR    = PROJECT_ROOT / "data" / "faiss"
FAISS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT.resolve()}")
print(f"PDF dir      : {PDF_DIR}")
print(f"FAISS dir    : {FAISS_DIR}")
print(f"PDFs found   : {[p.name for p in sorted(PDF_DIR.glob('*.pdf'))]}")

Project root : /home/ildebrando/code/ijesusjr/000_DS_Portfolio/02_prepsense
PDF dir      : data/pdfs
FAISS dir    : data/faiss
PDFs found   : ['emergency-supplies-cz.pdf', 'emergency-supplies-se.pdf', 'home-emergency-kit-be.pdf', 'putting-together-an-emergency-kit.pdf']


---
## 1. PDF Extraction

Three of the four source PDFs are image-based (no embedded text layer).
We use **PyMuPDF** for direct text extraction with an automatic **Tesseract OCR** fallback.

This cell shows exactly what raw content each page contains before cleaning.

In [2]:
import fitz  # pymupdf

def extract_page_text(page):
    """Try direct extraction first, fall back to OCR if empty."""
    text = page.get_text().strip()
    if text:
        return text, "direct"
    try:
        import pytesseract
        from PIL import Image
        mat = fitz.Matrix(3, 3)   # 3x zoom = ~216 DPI
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        return pytesseract.image_to_string(img, lang="eng").strip(), "OCR"
    except ImportError:
        return "", "no-ocr"

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    doc = fitz.open(str(pdf_path))
    print(f"\n{'='*55}")
    print(f"FILE : {pdf_path.name}")
    print(f"PAGES: {len(doc)}")
    for i, page in enumerate(doc):
        text, method = extract_page_text(page)
        words = len(text.split())
        preview = text[:80].replace("\n", " ")
        print(f"  p{i+1} [{method:<6}] {words:>4} words | {preview}...")
    doc.close()


FILE : emergency-supplies-cz.pdf
PAGES: 3
  p1 [OCR   ]  295 words | 72h22 Why 72 Hours Emergency Preparedness Vv Important Contacts Download  < Back...
  p2 [OCR   ]   98 words | T2h 2g Why 72 Hours Important Contacts © Download  Hand sanitizer  Toilet paper ...
  p3 [OCR   ]  114 words | T2h2 4 Why 72 Hours Important Contacts Download  can call.  112 Emergency Calls ...

FILE : emergency-supplies-se.pdf
PAGES: 2
  p1 [OCR   ]  620 words | Basics of home preparedness  Devise a plan detailing how to maintain access to w...
  p2 [OCR   ]  332 words | 72 hours has been used in information from government authorities in the United ...

FILE : home-emergency-kit-be.pdf
PAGES: 2
  p1 [OCR   ]  600 words | National Crisis Center >  AN EMERGENCY KIT AT HOME  An emergency kit is a set of...
  p2 [OCR   ]  536 words | NN IOS  « Some toys.  ° Acollar / harness and a leash.  « Near your backpack, a ...

FILE : putting-together-an-emergency-kit.pdf
PAGES: 1
  p1 [direct]  237 words | Store the it

---
## 2. Chunking

**Strategy:** 200-token chunks, 30-token overlap.

- Total content: ~2,800 words across 4 documents
- 200-token chunks → ~20 chunks → meaningful retrieval granularity
- 30-token overlap prevents sentences from being cut at boundaries
- The chunker also cleans OCR artifacts: nav menus, URLs, checkbox chars

In [3]:
from rag.chunker import extract_chunks, save_chunks, load_chunks

print("Extracting and chunking all PDFs...\n")
chunks = extract_chunks(str(PDF_DIR))
save_chunks(chunks, str(FAISS_DIR / "chunks.json"))

Extracting and chunking all PDFs...

  Processing: emergency-supplies-cz.pdf (3 pages)
    [p1] 288 words → 2 chunks
    [p2] 89 words → 1 chunks
    [p3] 78 words → 1 chunks
  Processing: emergency-supplies-se.pdf (2 pages)
    [p1] 602 words → 4 chunks
    [p2] 331 words → 2 chunks
  Processing: home-emergency-kit-be.pdf (2 pages)
    [p1] 589 words → 4 chunks
    [p2] 508 words → 3 chunks
  Processing: putting-together-an-emergency-kit.pdf (1 pages)
    [p1] 214 words → 2 chunks

Total chunks: 19
Chunks saved → data/faiss/chunks.json


In [4]:
# Inspect all chunks as a DataFrame
rows = [{
    "id":      c.chunk_id,
    "source":  c.source.split("(")[0].strip()[:32],
    "page":    c.page,
    "tokens":  c.tokens,
    "preview": c.text[:75] + "...",
} for c in chunks]

df_chunks = pd.DataFrame(rows)
print(f"Total chunks: {len(chunks)}\n")
print(df_chunks.to_string(index=False))

Total chunks: 19

 id                          source  page  tokens                                                                        preview
  0  Czech Republic Emergency Guide     1     200 72h22 Why 72 Hours Emergency Preparedness Vv Important Contacts Download Wo...
  1  Czech Republic Emergency Guide     1     118 go if it becomes necessary to leave home. At the same time, it is desirable...
  2  Czech Republic Emergency Guide     2      89 T2h 2g Why 72 Hours Important Contacts © Download Toilet paper and hygiene ...
  3  Czech Republic Emergency Guide     3      78 T2h2 4 Why 72 Hours Important Contacts Download 112 Emergency Calls 150 Fir...
  4          Sweden Emergency Guide     1     200 Basics of home preparedness Devise a plan detailing how to maintain access ...
  5          Sweden Emergency Guide     1     200 will be easier to co-operate and help each other. By joining an association...
  6          Sweden Emergency Guide     1     200 stop working. List, on paper,

In [5]:
# Per-source summary
print("Chunks per source:")
for source, grp in df_chunks.groupby("source"):
    total_tok = grp.tokens.sum()
    print(f"  {source}: {len(grp)} chunks | {total_tok} tokens")

Chunks per source:
  Belgium Emergency Kit Guide: 7 chunks | 1247 tokens
  Czech Republic Emergency Guide: 4 chunks | 485 tokens
  Netherlands Emergency Kit Guide: 2 chunks | 244 tokens
  Sweden Emergency Guide: 6 chunks | 1053 tokens


In [6]:
# Read one chunk in full to inspect quality
print("Full text of chunk 04 (Sweden guide, p1):\n")
print(chunks[4].text)

Full text of chunk 04 (Sweden guide, p1):

Basics of home preparedness Devise a plan detailing how to maintain access to water, food, communication and heating in your home for at least a You may need to adapt to your household's specific needs and circumstances. Is there anything else you or your loved ones require in a typical When you are prepared, society at large can focus on helping those who have the most difficulty coping on their own. If you ordinarily require special assistance, you are also entitled to assistance during a crisis; however, the level of service may vary, and it may take longer to receive. Test your abilities Seven days is a scenario based exercise with learning and practical tips. © Download or order in print at no cost (/sv/publikationer/seven-days/) Seven days is a scenario based exercise with learning and practical tips. © The digital exercise is only available in Swedish (/sv/rad-till-privatpersoner/ova/sju-dagar---ova-for-att-klara-en-kris-en-vecka/) Prep

---
## 3. Embedding

**Model:** `all-MiniLM-L6-v2`
- 384-dimensional dense vectors
- ~80MB download, fully local (CPU or GPU)
- No API calls, no rate limits, no cost
- Embeddings are L2-normalised so inner product = cosine similarity

First run downloads the model from HuggingFace (~80MB). Subsequent runs use cache.

In [7]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model: all-MiniLM-L6-v2")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dimension: {embed_model.get_embedding_dimension()}")

/home/ildebrando/.pyenv/versions/3.12.9/envs/prepsense/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2354.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [8]:
# Sanity check — similar sentences should score higher than dissimilar ones
test = [
    "How much water do I need in an emergency?",
    "An adult needs 2 to 3 litres of water per day.",
    "A battery-powered radio is important for receiving alerts.",
]
vecs = embed_model.encode(test, normalize_embeddings=True)

print("Cosine similarity sanity check:")
print(f"  Water question  ↔ water answer  : {vecs[0] @ vecs[1]:.3f}  ← should be HIGH")
print(f"  Water question  ↔ radio sentence : {vecs[0] @ vecs[2]:.3f}  ← should be LOW")

Cosine similarity sanity check:
  Water question  ↔ water answer  : 0.593  ← should be HIGH
  Water question  ↔ radio sentence : 0.147  ← should be LOW


In [9]:
# Embed all chunks
print(f"Embedding {len(chunks)} chunks...")
texts = [c.text for c in chunks]
embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(f"\nEmbeddings shape : {embeddings.shape}")
print(f"Memory usage     : {embeddings.nbytes / 1024:.1f} KB")

Embedding 19 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.52it/s]


Embeddings shape : (19, 384)
Memory usage     : 28.5 KB


---
## 4. FAISS Index

**Index type:** `IndexFlatIP` (flat inner product = cosine similarity on normalised vectors)

Exact search — no approximation. Correct choice for ~20 chunks.
For corpora >10k chunks you would switch to `IndexIVFFlat` or `IndexHNSWFlat`.

The index is persisted to `data/faiss/index.bin`.
After the first build, load it with `faiss.read_index()` in ~50ms.

In [10]:
import faiss

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"FAISS index built")
print(f"  Vectors   : {index.ntotal}")
print(f"  Dimension : {dim}")

index_path = str(FAISS_DIR / "index.bin")
faiss.write_index(index, index_path)
print(f"  Saved     → {index_path}")

FAISS index built
  Vectors   : 19
  Dimension : 384
  Saved     → data/faiss/index.bin


In [11]:
# Verify round-trip: reload and confirm counts match
index_reloaded = faiss.read_index(index_path)
chunks_reloaded = load_chunks(str(FAISS_DIR / "chunks.json"))

print("Round-trip check:")
print(f"  Vectors in index : {index_reloaded.ntotal}")
print(f"  Chunks in JSON   : {len(chunks_reloaded)}")
print(f"  Match            : {index_reloaded.ntotal == len(chunks_reloaded)}")

Round-trip check:
  Vectors in index : 19
  Chunks in JSON   : 19
  Match            : True


---
## 5. Retriever

`HavenRetriever` embeds a query and searches the FAISS index for the
most semantically similar chunks. Returns results with source label, page number,
and cosine similarity score (0–1).

This is the bridge between the user's question and the LLM context window.

In [ ]:
from rag.retriever import HavenRetriever

retriever = HavenRetriever.from_disk(
    index_path= str(FAISS_DIR / "index.bin"),
    meta_path=  str(FAISS_DIR / "chunks.json"),
)
print(f"Retriever ready | {index.ntotal} chunks indexed")

In [13]:
# Retrieval test 1 — water
query = "How much water do I need in my emergency kit?"
results = retriever.query(query, k=4)

print(f"Query: \"{query}\"\n")
for r in results:
    label = r.source.split("(")[0].strip()
    print(f"  [{r.score:.3f}] {label}, p{r.page}")
    print(f"           {r.text[:110]}...")
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2682.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: "How much water do I need in my emergency kit?"

  [0.565] Belgium Emergency Kit Guide, p2
           Acollar / harness and a leash. Near your backpack, a carrier adapted to your pet. Do you have exotic pets? Mak...

  [0.511] Belgium Emergency Kit Guide, p1
           National Crisis Center AN EMERGENCY KIT AT HOME An emergency kit is a set of items that are useful in a crisis...

  [0.509] Czech Republic Emergency Guide, p1
           go if it becomes necessary to leave home. At the same time, it is desirable to decide who and how will take ca...

  [0.482] Belgium Emergency Kit Guide, p1
           need to buy an expensive emergency kit. Most of your belongings are already at home, or can be shared with nei...



In [ ]:
# Retrieval test 2 — power outage scenario (key HAVEN use case)
query = "What should I prepare for in case of a power outage?"
results = retriever.query(query, k=4)

print(f"Query: \"{query}\"\n")
for r in results:
    label = r.source.split("(")[0].strip()
    print(f"  [{r.score:.3f}] {label}, p{r.page}")
    print(f"           {r.text[:110]}...")
    print()

In [15]:
# Retrieval test 3 — medication
query = "Why do I need medication in my emergency kit?"
results = retriever.query(query, k=3)

print(f"Query: \"{query}\"\n")
for r in results:
    label = r.source.split("(")[0].strip()
    print(f"  [{r.score:.3f}] {label}, p{r.page}")
    print(f"           {r.text[:110]}...")
    print()

Query: "Why do I need medication in my emergency kit?"

  [0.516] Belgium Emergency Kit Guide, p1
           need to buy an expensive emergency kit. Most of your belongings are already at home, or can be shared with nei...

  [0.497] Belgium Emergency Kit Guide, p1
           National Crisis Center AN EMERGENCY KIT AT HOME An emergency kit is a set of items that are useful in a crisis...

  [0.477] Belgium Emergency Kit Guide, p1
           GP, your insurance company). Copies of the identity cards of the members of your household. Apen and paper. A ...



In [16]:
# Score distribution — visual bar chart across all chunks
query = "emergency supplies checklist"
all_results = retriever.query(query, k=len(chunks))

print(f"Score distribution for: \"{query}\"\n")
print(f"  {'Score':<8} {'Source':<34} {'Page'}  Bar")
print(f"  {'-'*65}")
for r in all_results:
    bar   = "█" * int(r.score * 40)
    label = r.source.split("(")[0].strip()[:32]
    print(f"  {r.score:.3f}   {label:<34} p{r.page}  {bar}")

Score distribution for: "emergency supplies checklist"

  Score    Source                             Page  Bar
  -----------------------------------------------------------------
  0.610   Czech Republic Emergency Guide     p1  ████████████████████████
  0.607   Belgium Emergency Kit Guide        p1  ████████████████████████
  0.598   Netherlands Emergency Kit Guide    p1  ███████████████████████
  0.579   Belgium Emergency Kit Guide        p1  ███████████████████████
  0.557   Belgium Emergency Kit Guide        p1  ██████████████████████
  0.482   Sweden Emergency Guide             p1  ███████████████████
  0.482   Belgium Emergency Kit Guide        p2  ███████████████████
  0.475   Belgium Emergency Kit Guide        p2  ██████████████████
  0.450   Czech Republic Emergency Guide     p1  ██████████████████
  0.424   Netherlands Emergency Kit Guide    p1  ████████████████
  0.412   Belgium Emergency Kit Guide        p1  ████████████████
  0.406   Belgium Emergency Kit Guide        p2 

---
## 6. LLM Backends

Three backends, one interface — selected via `LLM_BACKEND` env var:

| `LLM_BACKEND` | When to use | Model | Cost |
|---------------|-------------|-------|------|
| `ollama` | Local development (GPU) | Mistral 7B | Free |
| `groq` | Free cloud deployment | Llama 3.1 8B | Free (14,400 req/day) |
| `anthropic` | Paid fallback | Claude Haiku | ~$0.25/1M tokens |

The prompt template is identical across all backends.
Add your keys to `.env`: `GROQ_API_KEY=...` or `ANTHROPIC_API_KEY=...`

In [17]:
BACKEND = os.getenv("LLM_BACKEND", None)  # reads "groq" from .env

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()
print(f"LLM_BACKEND : {os.getenv('LLM_BACKEND')}")
print(f"GROQ_API_KEY: {'set' if os.getenv('GROQ_API_KEY') else 'NOT SET'}")

LLM_BACKEND : groq
GROQ_API_KEY: set


In [ ]:
from rag.llm import HavenLLM

print("Backend availability:\n")
for backend in ["ollama", "groq", "anthropic"]:
    llm_check = HavenLLM(backend=backend)
    status = "ready" if llm_check.is_available() else "not configured"
    icon   = "✅" if llm_check.is_available() else "❌"
    print(f"  {icon} {backend:<12} {status}")

print()
ollama_llm = HavenLLM(backend="ollama")
if ollama_llm.is_available():
    models = ollama_llm.list_ollama_models()
    print(f"Ollama models available: {models}")

In [ ]:
# Auto-detect best available backend
BACKEND = os.getenv("LLM_BACKEND", None)

if BACKEND is None:
    for candidate in ["groq", "ollama", "anthropic"]:
        if HavenLLM(backend=candidate).is_available():
            BACKEND = candidate
            break

if BACKEND:
    llm = HavenLLM(backend=BACKEND)
    print(f"Active backend: {BACKEND}")
else:
    llm = None
    print("No LLM backend available.")
    print("  → Add GROQ_API_KEY to .env  (free at console.groq.com)")
    print("  → Or run: ollama pull mistral && ollama serve")

In [24]:
# Smoke test — simple direct call without retriever
if llm is not None:
    from rag.retriever import RetrievedChunk
    fake_chunk = RetrievedChunk(
        chunk_id=0,
        text="An adult needs about 3 litres of water per day for drinking and basic hygiene. "
             "Store water in sealed, non-perishable containers.",
        source="Netherlands Emergency Kit Guide (denkvooruit.nl)",
        page=1,
        score=0.92,
    )
    answer = llm.answer(
        question="How much water do I need per person per day?",
        retrieved_chunks=[fake_chunk],
        gaps=[],
    )
    print("LLM smoke test answer:")
    print("-" * 50)
    print(answer)
else:
    print("Skipping — no LLM backend configured")

LLM smoke test answer:
--------------------------------------------------
Based on the provided knowledge base, the recommended daily water intake per adult is 3 litres for drinking and basic hygiene. 

To determine how much water you need per person per day, refer to the Netherlands Emergency Kit Guide [Source: Netherlands Emergency Kit Guide, Page 1]. 

Since your current kit appears complete, you don't have any gaps to address. However, it's essential to note that the guide recommends storing water in sealed, non-perishable containers. If you're not already doing so, consider adding this to your kit to ensure you have a reliable water supply in case of an emergency.


---
## 7. Full Pipeline

`HavenPipeline` wires retriever → LLM → cited answer in one call.

```python
pipeline.ask(question, gaps, k) → RAGResponse(response, chunks, sources)
```

The response always includes `[Source: ..., Page X]` citations
grounded in the actual retrieved chunks.

In [ ]:
from rag.pipeline import HavenPipeline

pipeline = HavenPipeline.from_disk(
    index_path=   str(FAISS_DIR / "index.bin"),
    meta_path=    str(FAISS_DIR / "chunks.json"),
    backend=      BACKEND,
    ollama_model= "mistral",
)
print("Pipeline ready")

In [26]:
# Pipeline query 1 — item justification
if llm is not None:
    answer = pipeline.ask(
        question="Why do I need a battery-powered radio in my emergency kit?",
        gaps=[],
        k=4,
    )
    pipeline.print_answer(answer)
else:
    print("Skipping — no LLM backend configured")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3166.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Q: Why do I need a battery-powered radio in my emergency kit?

You don't have any kit gaps detected, but I'll provide the information on why a battery-powered radio is essential in your emergency kit.

According to the Belgium Emergency Kit Guide [Source: Belgium Emergency Kit Guide (crisiscenter.be), Page 2], a battery-powered or wind-up radio is recommended to listen to the media in the event of a power outage. This is crucial for staying informed about the situation, receiving important updates, and potentially receiving instructions from authorities.

In the event of a crisis, you may be without electricity, and a battery-powered radio will allow you to stay connected and receive vital information. This is especially important for staying safe and making informed decisions about your situation.

Practical advice: Make sure to include a battery-powered radio in your emergency kit, and consider including spare batteries to ensure it remains functional during an extended power outage

In [27]:
# Pipeline query 2 — power outage scenario
if llm is not None:
    answer = pipeline.ask(
        question="What should I prepare for a 72-hour power outage?",
        gaps=[],
        k=4,
    )
    pipeline.print_answer(answer)
else:
    print("Skipping — no LLM backend configured")


Q: What should I prepare for a 72-hour power outage?

Based on the provided EU emergency preparedness guidance, I recommend the following items to prepare for a 72-hour power outage:

1. **Light source**: A flashlight, preferably dynamo-powered [Source: Belgium Emergency Kit Guide, Page 2]. If you prefer a battery-powered lamp, bring spare batteries.
2. **Alternative heating**: Blankets for extra warmth [Source: Belgium Emergency Kit Guide, Page 2].
3. **Food and water**: Ensure you have enough non-perishable food and bottled drinking water for 72 hours. An adult can manage with 2 litres of water per day [Source: Czech Republic Emergency Guide, Page 1].
4. **Communication**: A battery-powered or wind-up radio to stay informed [Source: Belgium Emergency Kit Guide, Page 2].
5. **Power bank and spare batteries**: A charged power bank and spare batteries to keep your devices charged [Source: Czech Republic Emergency Guide, Page 1].

These items will help you and your household members sta

---
## 8. Gap-Aware Queries

The key differentiator of HAVEN RAG.

The LLM receives **both** the retrieved guidance AND the user's **current kit gaps**
so answers are personalised to the user's specific situation, not generic.

**Generic answer:** "You need 3 litres of water per person per day."
**Gap-aware answer:** "You currently have 2L but need 9L (78% short). According to
the Dutch guide, store water in sealed containers. Given your gap, prioritise..."


In [29]:
from core.inventory_analyzer import KitItem
import inspect
print(inspect.signature(KitItem))

(name: str, category: str, quantity: float, unit: str, eu_recommended: float, expiry_date: Optional[datetime.date] = None, notes: str = '') -> None


In [30]:
from core.inventory_analyzer import KitItem, analyze_inventory

# Simulate a realistic partially-stocked kit
sample_kit = [
    KitItem(name="Drinking water",        category="water",   quantity=2.0,  eu_recommended=9.0,  unit="liters", expiry_date=None),
    KitItem(name="Non-perishable food",   category="food",    quantity=1.0,  eu_recommended=3.0,  unit="days",   expiry_date=None),
    KitItem(name="First aid kit",         category="meds",    quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Regular medication",    category="meds",    quantity=0.0,  eu_recommended=7.0,  unit="days",   expiry_date=None),
    KitItem(name="Battery-powered radio", category="comms",   quantity=0.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Flashlight",            category="light",   quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Cash",                  category="cash",    quantity=20.0, eu_recommended=70.0, unit="EUR",    expiry_date=None),
    KitItem(name="Hand sanitizer",        category="hygiene", quantity=0.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
]
inv_report = analyze_inventory(sample_kit)

print(f"Kit analysis:")
print(f"  Gaps        : {len(inv_report.gaps)}")
print(f"  Critical    : {sum(1 for g in inv_report.gaps if g.priority == 'HIGH')}")
print(f"  Gap score   : {inv_report.total_gap_score}/100")
print()
print("Active gaps:")
for g in inv_report.gaps:
    bar = "█" * int(g.gap_pct / 5)
    print(f"  [{g.priority:<6}] {g.name:<25} {g.current:.1f}/{g.recommended:.1f} {g.unit:<6} {bar}")

Kit analysis:
  Gaps        : 6
  Critical    : 3
  Gap score   : 36/100

Active gaps:
  [HIGH  ] Regular medication        0.0/7.0 days   ████████████████████
  [HIGH  ] Drinking water            2.0/9.0 liters ███████████████
  [HIGH  ] Non-perishable food       1.0/3.0 days   █████████████
  [MEDIUM] Battery-powered radio     0.0/1.0 units  ████████████████████
  [LOW   ] Hand sanitizer            0.0/1.0 units  ████████████████████
  [LOW   ] Cash                      20.0/70.0 EUR    ██████████████


In [31]:
# Gap-aware query 1 — water storage
if llm is not None:
    answer = pipeline.ask(
        question="Why do I need more water and how should I store it safely?",
        gaps=inv_report.gaps,
        k=4,
    )
    pipeline.print_answer(answer)
else:
    print("Skipping — no LLM backend configured")


Q: Why do I need more water and how should I store it safely?

Based on the provided EU emergency preparedness guidance, you need more water to meet the recommended 3-10 liters per person per day for 3 days. 

According to the Belgium Emergency Kit Guide [Source: Belgium Emergency Kit Guide, Page 2], an adult drinks between 1.5 and 2 liters of water per day, and the rest can be used for washing, using the toilet, and cooking. Considering you need 9.0 liters and have 2.0 liters, you are missing 7.0 liters.

To store water safely, follow these tips:

1. Use sealable containers, such as canisters [Source: Czech Republic Emergency Guide, Page 1].
2. Keep the containers in a dry place, away from potential flooding areas like cellars or garages [Source: Belgium Emergency Kit Guide, Page 2].
3. Check the containers regularly to ensure they are not damaged or contaminated.

To address your current kit gaps, I recommend adding 7.0 liters of bottled drinking water to your emergency kit.

--- So

In [ ]:
# Gap-aware query 2 — the HAVEN signature scenario
if llm is not None:
    answer = pipeline.ask(
        question="How long will my current kit last if power is out for 72 hours?",
        gaps=inv_report.gaps,
        k=4,
    )
    pipeline.print_answer(answer)
else:
    print("Skipping — no LLM backend configured")

In [33]:
# Gap-aware query 3 — medication gap
if llm is not None:
    answer = pipeline.ask(
        question="I have no medication stock. What do the official EU guides recommend?",
        gaps=inv_report.gaps,
        k=3,
    )
    pipeline.print_answer(answer)
else:
    print("Skipping — no LLM backend configured")


Q: I have no medication stock. What do the official EU guides recommend?

Based on the official EU guides, it is recommended to keep a reserve supply of essential medication for at least 3 days in your emergency kit. 

According to the Belgium Emergency Kit Guide [Source: Belgium Emergency Kit Guide (crisiscenter.be), Page 1], if you or a member of your family need to take essential medication, you should keep a reserve supply (for at least 3 days) in your bag. It is recommended to ask your pharmacist or GP for a list of medicines.

Given your current kit gaps, I would advise you to prioritize acquiring a 7-day supply of your regular medication. This is a HIGH priority item, as you currently have 0.0 days of medication stock, which is 100% missing.

To address this gap, I recommend the following:

1. Contact your pharmacist or GP to obtain a list of your essential medications.
2. Ask your pharmacist or GP to recommend a 7-day supply of your regular medication.
3. Acquire the recommend

---
## 9. Multi-Source Comparison

Check that all four national guides contribute to retrieval across different topics.
This validates the knowledge base has good coverage and no single document dominates.

In [34]:
queries = [
    "How much water per person per day?",
    "What documents should I keep in my emergency bag?",
    "How should I prepare for evacuation?",
    "What food should I store for emergencies?",
    "How do I signal for help if I am in danger?",
    "What should I do during a power outage?",
]

print(f"{'Query':<48}  Top 3 sources (score)")
print("-" * 95)

for q in queries:
    results = retriever.query(q, k=3)
    sources = []
    for r in results:
        label = r.source.split("(")[0].strip()
        # Abbreviate country names
        label = (label
            .replace("Czech Republic Emergency Guide", "CZ")
            .replace("Sweden Emergency Guide", "SE")
            .replace("Belgium Emergency Kit Guide", "BE")
            .replace("Netherlands Emergency Kit Guide", "NL")
        )
        sources.append(f"{label}({r.score:.2f})")
    print(f"{q[:46]:<48}  {' | '.join(sources)}")

Query                                             Top 3 sources (score)
-----------------------------------------------------------------------------------------------
How much water per person per day?                BE(0.33) | CZ(0.29) | SE(0.23)
What documents should I keep in my emergency b    BE(0.66) | NL(0.58) | BE(0.57)
How should I prepare for evacuation?              BE(0.58) | BE(0.58) | BE(0.56)
What food should I store for emergencies?         NL(0.63) | BE(0.60) | BE(0.55)
How do I signal for help if I am in danger?       SE(0.53) | SE(0.42) | SE(0.42)
What should I do during a power outage?           BE(0.48) | BE(0.48) | BE(0.43)


---
## 10. Summary

In [ ]:
from pathlib import Path

print("HAVEN RAG — Complete")
print("=" * 50)
print(f"  Source PDFs    : {len(list(PDF_DIR.glob('*.pdf')))}")
print(f"  Total chunks   : {len(chunks)}")
print(f"  Embedding model: all-MiniLM-L6-v2 (384 dim)")
print(f"  FAISS index    : IndexFlatIP (exact cosine, {index.ntotal} vectors)")
index_size = Path(index_path).stat().st_size / 1024
print(f"  Index size     : {index_size:.1f} KB")
print(f"  LLM backend    : {BACKEND or 'not configured'}")
print()

from collections import Counter
source_counts = Counter(c.source.split("(")[0].strip() for c in chunks)
print("Chunks per source:")
for source, count in sorted(source_counts.items()):
    print(f"  {source}: {count} chunks")

print()
print("Tasks complete:")
print("  [x] PDF extraction (direct + OCR fallback)")
print("  [x] Chunking (200 tokens / 30 overlap)")
print("  [x] Embedding (all-MiniLM-L6-v2, local)")
print("  [x] FAISS index (built + persisted to disk)")
print("  [x] Retriever (semantic search, top-k)")
print("  [x] LLM backends (Ollama / Groq / Anthropic)")
print("  [x] Full pipeline (retriever → LLM → cited answer)")
print("  [x] Gap-aware queries (personalised to kit inventory)")